# GRU Colab Notebook 

In [ ]:
#Import files from Google Drive (add your own paths)
import os
from google.colab import drive
drive.mount("/content/gdrive", force_remount=True)
os.chdir('/content/gdrive/MyDrive/ColabNotebooks/GRU')
print(os.getcwd())
snapshot_dir = '/content/gdrive/MyDrive/ColabNotebooks/GRU/gru_snapshots'
os.makedirs(snapshot_dir, exist_ok=True)
print(f"Snapshots will be saved to: {snapshot_dir}")

In [ ]:
!unzip /content/gdrive/MyDrive/ColabNotebooks/guitar_dataset.zip -d /content/guitar_dataset

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import librosa
import librosa.display
import matplotlib.pyplot as plt
import soundfile as sf
import numpy as np
from IPython.display import Audio, display
import os
import time


from gru_model import GRUModel
from gru_training import GRUTrainer
from gru_paired_audio_data import PairedGRUDataset
from stft_loss import MRSTFTLoss
from gru_logging import TensorboardLogger

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
#Dataset Settings
SAMPLING_RATE = 16000
DATASET_FILE = '/content/guitar_dataset/gru_paired_dataset.npz'
CLEAN_DIR = '/content/guitar_dataset/clean_guitar'
PROCESSED_DIR = '/content/guitar_dataset/overdrive_guitar'

# GRU Parameters

SEQUENCE_LENGTH = 8192 # L
HIDDEN_SIZE = 128      # Hidden Units
NUM_LAYERS = 2         # GRU layers
BIDIRECTIONAL = False
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
EPOCHS = 90
SNAPSHOT_INTERVAL = 800

In [ ]:
dataset = PairedGRUDataset(
    dataset_file=DATASET_FILE,
    item_length=SEQUENCE_LENGTH,
    clean_dir=CLEAN_DIR,
    processed_dir=PROCESSED_DIR,
    target_length=SEQUENCE_LENGTH,
    sampling_rate=SAMPLING_RATE,
    device=device
)


In [ ]:
#Initialize Model
model = GRUModel(
    input_size=1,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    output_size=1,
    bidirectional=BIDIRECTIONAL,
    device=device)

In [ ]:
# Initiaize Logger and Trainer
logger = TensorboardLogger(log_interval=50, validation_interval=100)

trainer = GRUTrainer(
    model=model,
    dataset=dataset,
    lr=LEARNING_RATE,
    gradient_clipping=1.0,
    logger=logger,
    snapshot_path=snapshot_dir,
    snapshot_interval=SNAPSHOT_INTERVAL,
    device=device
)

logger.trainer = trainer
model = trainer.model
optimizer = trainer.optimizer
scheduler = trainer.scheduler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Starting GRU Training...")
trainer.train(batch_size=BATCH_SIZE, epochs=EPOCHS)

In [ ]:
# Load latest snapshot
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_path = '/content/gdrive/MyDrive/ColabNotebooks/GRU/gru_snapshots/snapshot_2026-03-09_20-44-25.pth'

ckpt = torch.load(model_path, map_location=device)

model.load_state_dict(ckpt['model_state_dict'])
model = model.to(device)
model.eval()

print(f"Model restored from: {model_path}")
print("Status: Set to eval mode.")

In [ ]:
import torch
import torchaudio
from IPython.display import Audio


input_wav_path = '/content/gdrive/MyDrive/ColabNotebooks/2-0.wav'
output_wav_path = '/content/gdrive/MyDrive/ColabNotebooks/GRU/gru_generated_audio/gru_processed_2-0.wav'
print(f"Processing: {input_wav_path}")


waveform, sample_rate = torchaudio.load(input_wav_path)
waveform = waveform.to(device).float()


if waveform.shape[0] > 1:
    waveform = torch.mean(waveform, dim=0, keepdim=True)



with torch.no_grad():

    input_tensor = waveform.transpose(0, 1).unsqueeze(0).contiguous()

    with torch.backends.cudnn.flags(enabled=False):   
        output_tensor = model(input_tensor)
    output_waveform = output_tensor.squeeze(0).transpose(0, 1)


output_waveform = output_waveform.cpu()
torchaudio.save(output_wav_path, output_waveform, sample_rate)


print(f"Success! Saved to: {output_wav_path}")

Audio(output_wav_path)